# 02 — Data Quality, Cleaning & Feature Engineering

This notebook validates the raw weather data loaded into DuckDB, applies cleaning logic, builds model-ready features, and stores the final feature table in DuckDB.

The output of this notebook is:

`analytics.model_features`

This table is used by the modeling notebook and by the automated pipeline.

## 1. Imports & Setup


In [3]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.db import run_query, get_connection

from src.quality_checks import (
    check_missing_values,
    check_duplicate_rows,
    check_duplicate_city_dates,
    check_date_coverage,
    check_missing_dates,
    check_column_consistency,
    check_weather_ranges,
)

from src.cleaning import clean_data

from src.features import (
    build_features,
    get_feature_columns,
    get_target_columns,
)

## 2. Load Raw Data from DuckDB


In [5]:
historical_df = run_query("SELECT * FROM raw.historical").copy()
forecast_df = run_query("SELECT * FROM raw.forecast").copy()

historical_df["time"] = pd.to_datetime(historical_df["time"])
forecast_df["time"] = pd.to_datetime(forecast_df["time"])

print("Historical shape:", historical_df.shape)
print("Forecast shape:", forecast_df.shape)

display(historical_df.head())
display(forecast_df.head())

Historical shape: (10960, 9)
Forecast shape: (35, 9)


,time,temperature_2m_max,precipitation_sum,wind_speed_10m_max,relative_humidity_2m_mean,cloud_cover_mean,apparent_temperature_max,sunshine_duration,city
0,2020-01-01,11.4,0.0,11.4,90,30,8.8,31885.13,Baku
1,2020-01-02,8.6,0.0,33.0,88,50,3.2,30162.88,Baku
2,2020-01-03,7.6,1.7,24.1,83,96,3.3,1147.69,Baku
3,2020-01-04,8.4,0.2,22.7,83,98,4.6,1300.71,Baku
4,2020-01-05,7.7,2.6,12.6,91,100,6.4,0.00,Baku


,time,temperature_2m_max,precipitation_sum,wind_speed_10m_max,relative_humidity_2m_mean,cloud_cover_mean,apparent_temperature_max,sunshine_duration,city
0,2026-04-25,14.8,0.0,30.2,80,78,13.7,29133.25,Baku
1,2026-04-26,16.0,0.0,25.5,81,44,13.6,46698.68,Baku
2,2026-04-27,18.8,0.2,50.8,78,83,18.0,26936.03,Baku
3,2026-04-28,14.5,0.0,44.1,54,85,8.7,30387.62,Baku
4,2026-04-29,15.7,0.3,12.6,71,96,14.4,13624.88,Baku


## 3. Historical Data Quality Checks


In [7]:
print("Historical data quality checks")

print("\nMissing values:")
display(check_missing_values(historical_df))

print("\nDuplicate rows:")
print(check_duplicate_rows(historical_df))

print("\nDuplicate city-date records:")
print(check_duplicate_city_dates(historical_df))

print("\nWeather range violations:")
print(check_weather_ranges(historical_df))

Historical data quality checks

Missing values:


{'check': 'missing_values',
 'status': 'PASS',
 'details': {'time': 0,
  'temperature_2m_max': 0,
  'precipitation_sum': 0,
  'wind_speed_10m_max': 0,
  'relative_humidity_2m_mean': 0,
  'cloud_cover_mean': 0,
  'apparent_temperature_max': 0,
  'sunshine_duration': 0,
  'city': 0}}


Duplicate rows:
{'check': 'duplicate_rows', 'status': 'PASS', 'details': '0 duplicate rows'}

Duplicate city-date records:
{'check': 'duplicate_city_dates', 'status': 'PASS', 'details': '0 duplicate city-date records'}

Weather range violations:
{'check': 'weather_ranges', 'status': 'PASS', 'details': {'temperature_2m_max': 0, 'precipitation_sum': 0, 'wind_speed_10m_max': 0, 'relative_humidity_2m_mean': 0, 'cloud_cover_mean': 0, 'apparent_temperature_max': 0, 'sunshine_duration': 0}}


## 4. Forecast Data Quality Checks


In [9]:
print("Forecast data quality checks")

print("\nMissing values:")
display(check_missing_values(forecast_df))

print("\nDuplicate rows:")
print(check_duplicate_rows(forecast_df))

print("\nDuplicate city-date records:")
print(check_duplicate_city_dates(forecast_df))

print("\nWeather range violations:")
print(check_weather_ranges(forecast_df))

Forecast data quality checks

Missing values:


{'check': 'missing_values',
 'status': 'PASS',
 'details': {'time': 0,
  'temperature_2m_max': 0,
  'precipitation_sum': 0,
  'wind_speed_10m_max': 0,
  'relative_humidity_2m_mean': 0,
  'cloud_cover_mean': 0,
  'apparent_temperature_max': 0,
  'sunshine_duration': 0,
  'city': 0}}


Duplicate rows:
{'check': 'duplicate_rows', 'status': 'PASS', 'details': '0 duplicate rows'}

Duplicate city-date records:
{'check': 'duplicate_city_dates', 'status': 'PASS', 'details': '0 duplicate city-date records'}

Weather range violations:
{'check': 'weather_ranges', 'status': 'PASS', 'details': {'temperature_2m_max': 0, 'precipitation_sum': 0, 'wind_speed_10m_max': 0, 'relative_humidity_2m_mean': 0, 'cloud_cover_mean': 0, 'apparent_temperature_max': 0, 'sunshine_duration': 0}}


## 5. Coverage and Consistency Checks


In [11]:
print("Historical city/date coverage:")
display(check_date_coverage(historical_df))

print("\nHistorical missing dates by city:")
print(check_missing_dates(historical_df))

print("\nHistorical city counts:")
display(historical_df["city"].value_counts())

print("\nForecast city counts:")
display(forecast_df["city"].value_counts())

print("\nHistorical vs forecast column consistency:")
check_column_consistency(historical_df, forecast_df)

Historical city/date coverage:


{'check': 'date_coverage',
 'status': 'PASS',
 'details': [{'city': 'Baku',
   'min': Timestamp('2020-01-01 00:00:00'),
   'max': Timestamp('2025-12-31 00:00:00'),
   'count': 2192},
  {'city': 'Gabala',
   'min': Timestamp('2020-01-01 00:00:00'),
   'max': Timestamp('2025-12-31 00:00:00'),
   'count': 2192},
  {'city': 'Guba',
   'min': Timestamp('2020-01-01 00:00:00'),
   'max': Timestamp('2025-12-31 00:00:00'),
   'count': 2192},
  {'city': 'Lankaran',
   'min': Timestamp('2020-01-01 00:00:00'),
   'max': Timestamp('2025-12-31 00:00:00'),
   'count': 2192},
  {'city': 'Shaki',
   'min': Timestamp('2020-01-01 00:00:00'),
   'max': Timestamp('2025-12-31 00:00:00'),
   'count': 2192}]}


Historical missing dates by city:
{'check': 'missing_dates', 'status': 'PASS', 'details': {'Baku': 0, 'Gabala': 0, 'Guba': 0, 'Lankaran': 0, 'Shaki': 0}}

Historical city counts:


city
Baku        2192
Gabala      2192
Guba        2192
Lankaran    2192
Shaki       2192
Name: count, dtype: int64


Forecast city counts:


city
Baku        7
Gabala      7
Guba        7
Lankaran    7
Shaki       7
Name: count, dtype: int64


Historical vs forecast column consistency:


{'check': 'column_consistency',
 'status': 'PASS',
 'details': {'same_columns': True, 'only_in_first': [], 'only_in_second': []}}

## 6. Data Quality Summary

The historical and forecast datasets were evaluated using automated quality checks, including missing values, duplicate rows, duplicate city-date records, date coverage, date continuity, city coverage, column consistency, and realistic weather ranges.

### Key Findings

- **Missing Values:**  
  No missing values were detected in either historical or forecast datasets across all features.

- **Duplicates:**  
  No duplicate rows or duplicate city-date records were found, confirming data uniqueness.

- **Date Coverage:**  
  Each city contains a complete and consistent daily time series:
  - Historical data spans from **2020-01-01 to 2025-12-31**
  - Each city has **2192 observations**, indicating full coverage
  - Forecast data includes **7 days for each city**

- **Date Continuity:**  
  No missing dates were detected for any city, ensuring a continuous time series without gaps.

- **City Coverage:**  
  All selected cities (Baku, Gabala, Guba, Lankaran, Shaki) are consistently represented in both historical and forecast datasets.

- **Column Consistency:**  
  Historical and forecast datasets share identical feature columns, ensuring compatibility for downstream processing.

- **Range Validation:**  
  No values were found outside predefined realistic physical ranges for weather variables, indicating high data reliability.

### Data Cleaning Decisions

Since no significant issues were detected:
- No imputation was required for missing values
- No duplicate removal was necessary
- No outlier correction or filtering was applied
- No date reconstruction or interpolation was needed

### Conclusion

The dataset is **fully clean, complete, and consistent**, requiring minimal preprocessing.  
It is therefore **highly suitable for feature engineering, modeling, and time-series analysis**, while preserving data integrity and temporal structure.

## 7. Clean Historical Data


### 7.1 — apply cleaning

In [15]:
clean_df = clean_data(historical_df)

print("Raw historical shape:", historical_df.shape)
print("Clean historical shape:", clean_df.shape)

clean_df.head()

Raw historical shape: (10960, 9)
Clean historical shape: (10960, 9)


,time,temperature_2m_max,precipitation_sum,wind_speed_10m_max,relative_humidity_2m_mean,cloud_cover_mean,apparent_temperature_max,sunshine_duration,city
0,2020-01-01,11.4,0.0,11.4,90,30,8.8,31885.13,Baku
1,2020-01-02,8.6,0.0,33.0,88,50,3.2,30162.88,Baku
2,2020-01-03,7.6,1.7,24.1,83,96,3.3,1147.69,Baku
3,2020-01-04,8.4,0.2,22.7,83,98,4.6,1300.71,Baku
4,2020-01-05,7.7,2.6,12.6,91,100,6.4,0.00,Baku


### 7.2 — validate cleaned data

In [18]:
print("Cleaned data validation")

print("\nMissing values:")
display(clean_df.isna().sum())

print("\nDuplicate rows:")
print(clean_df.duplicated().sum())

print("\nDuplicate city-date records:")
print(clean_df.duplicated(subset=["city", "time"]).sum())

print("\nDate continuity check:")
display(clean_df.groupby("city")["time"].diff().value_counts())

Cleaned data validation

Missing values:


time                         0
temperature_2m_max           0
precipitation_sum            0
wind_speed_10m_max           0
relative_humidity_2m_mean    0
cloud_cover_mean             0
apparent_temperature_max     0
sunshine_duration            0
city                         0
dtype: int64


Duplicate rows:
0

Duplicate city-date records:
0

Date continuity check:


time
1 days    10955
Name: count, dtype: int64

## 8. Feature Engineering


### 8.1 — build features

In [25]:
feature_df, city_encoder = build_features(clean_df)

print("Feature dataframe shape:", feature_df.shape)
feature_df.head()

Feature dataframe shape: (10925, 44)


,time,temperature_2m_max,precipitation_sum,wind_speed_10m_max,relative_humidity_2m_mean,cloud_cover_mean,apparent_temperature_max,sunshine_duration,city,month,...,wind_7d_avg,humidity_7d_avg,humidity_14d_avg,cloud_cover_7d_avg,rainy_days_7d,temperature_trend_1d,humidity_trend_1d,wind_trend_1d,precipitation_trend_1d,city_encoded
0,2020-01-08,10.1,0.0,14.2,84,51,7.3,29491.43,Baku,1,...,19.828571,87.428571,87.428571,69.714286,2.0,0.3,-2.0,-11.3,0.0,0
1,2020-01-09,9.2,0.4,30.2,88,97,4.2,9599.62,Baku,1,...,20.228571,86.571429,87.000000,72.714286,2.0,-0.9,4.0,16.0,0.4,0
2,2020-01-10,7.3,1.9,24.0,87,82,3.6,23398.28,Baku,1,...,19.828571,86.571429,87.111111,79.428571,2.0,-1.9,-1.0,-6.2,1.5,0
3,2020-01-11,6.1,2.1,16.3,88,91,4.2,0.00,Baku,1,...,19.814286,87.142857,87.100000,77.428571,2.0,-1.2,1.0,-7.7,0.2,0
4,2020-01-12,8.5,0.0,28.3,83,59,2.6,31925.68,Baku,1,...,18.900000,87.857143,87.181818,76.428571,3.0,2.4,-5.0,12.0,-2.1,0


### 8.2 — inspect features and targets

In [29]:
feature_cols = get_feature_columns()
target_cols = get_target_columns()

print("Feature columns:")
for col in feature_cols:
    print(f"- {col}")

print("\nTarget columns:")
for col in target_cols:
    print(f"- {col}")

print("\nMissing values after feature engineering:")
display(feature_df[feature_cols + target_cols].isna().sum())

Feature columns:
- apparent_temperature_max
- sunshine_duration
- city_encoded
- month
- day_of_month
- day_sin
- day_cos
- comfort_gap
- sunshine_ratio
- temperature_lag_1
- temperature_lag_3
- temperature_lag_7
- precipitation_lag_1
- precipitation_lag_3
- precipitation_lag_7
- wind_lag_1
- wind_lag_3
- humidity_lag_1
- humidity_lag_3
- temperature_3d_avg
- temperature_7d_avg
- temperature_14d_avg
- precipitation_3d_sum
- precipitation_7d_sum
- precipitation_14d_sum
- wind_3d_avg
- wind_7d_avg
- humidity_7d_avg
- humidity_14d_avg
- cloud_cover_7d_avg
- rainy_days_7d
- temperature_trend_1d
- humidity_trend_1d
- wind_trend_1d
- precipitation_trend_1d

Target columns:
- temperature_2m_max
- precipitation_sum
- wind_speed_10m_max
- relative_humidity_2m_mean
- cloud_cover_mean

Missing values after feature engineering:


apparent_temperature_max     0
sunshine_duration            0
city_encoded                 0
month                        0
day_of_month                 0
day_sin                      0
day_cos                      0
comfort_gap                  0
sunshine_ratio               0
temperature_lag_1            0
temperature_lag_3            0
temperature_lag_7            0
precipitation_lag_1          0
precipitation_lag_3          0
precipitation_lag_7          0
wind_lag_1                   0
wind_lag_3                   0
humidity_lag_1               0
humidity_lag_3               0
temperature_3d_avg           0
temperature_7d_avg           0
temperature_14d_avg          0
precipitation_3d_sum         0
precipitation_7d_sum         0
precipitation_14d_sum        0
wind_3d_avg                  0
wind_7d_avg                  0
humidity_7d_avg              0
humidity_14d_avg             0
cloud_cover_7d_avg           0
rainy_days_7d                0
temperature_trend_1d         0
humidity

## 9. Store Model-Ready Dataset in DuckDB


### 9.1 — store table

In [34]:
with get_connection() as conn:
    conn.execute("CREATE SCHEMA IF NOT EXISTS analytics;")
    conn.register("feature_df_view", feature_df)

    conn.execute("""
        CREATE OR REPLACE TABLE analytics.model_features AS
        SELECT * FROM feature_df_view;
    """)

print("analytics.model_features table created.")

analytics.model_features table created.


### 9.2 — verify table

In [37]:
model_features_summary = run_query("""
SELECT
    MIN(time) AS min_date,
    MAX(time) AS max_date,
    COUNT(*) AS rows
FROM analytics.model_features
""")

display(model_features_summary)

run_query("""
SELECT *
FROM analytics.model_features
LIMIT 5
""")

,min_date,max_date,rows
0,2020-01-08,2025-12-31,10925


,time,temperature_2m_max,precipitation_sum,wind_speed_10m_max,relative_humidity_2m_mean,cloud_cover_mean,apparent_temperature_max,sunshine_duration,city,month,...,wind_7d_avg,humidity_7d_avg,humidity_14d_avg,cloud_cover_7d_avg,rainy_days_7d,temperature_trend_1d,humidity_trend_1d,wind_trend_1d,precipitation_trend_1d,city_encoded
0,2020-01-08,10.1,0.0,14.2,84,51,7.3,29491.43,Baku,1,...,19.828571,87.428571,87.428571,69.714286,2.0,0.3,-2.0,-11.3,0.0,0
1,2020-01-09,9.2,0.4,30.2,88,97,4.2,9599.62,Baku,1,...,20.228571,86.571429,87.000000,72.714286,2.0,-0.9,4.0,16.0,0.4,0
2,2020-01-10,7.3,1.9,24.0,87,82,3.6,23398.28,Baku,1,...,19.828571,86.571429,87.111111,79.428571,2.0,-1.9,-1.0,-6.2,1.5,0
3,2020-01-11,6.1,2.1,16.3,88,91,4.2,0.00,Baku,1,...,19.814286,87.142857,87.100000,77.428571,2.0,-1.2,1.0,-7.7,0.2,0
4,2020-01-12,8.5,0.0,28.3,83,59,2.6,31925.68,Baku,1,...,18.900000,87.857143,87.181818,76.428571,3.0,2.4,-5.0,12.0,-2.1,0


## 10. Final Summary

The raw historical data was validated, cleaned, and transformed into a model-ready feature table.

The final dataset includes:

- original target weather variables
- calendar features
- lag features
- rolling trend features
- encoded city information

The prepared dataset was stored in DuckDB as:

`analytics.model_features`

This table is now ready for model training and evaluation.